In [1]:
import tvm
from tvm import relay
import numpy as np
from tvm.contrib.download import download_testdata
from tvm.contrib import graph_executor


import torch
import torchvision
from tvm import relay
from tvm.contrib import cc


# Load SqueezeNet
squeezenet = torch.hub.load('pytorch/vision:v0.14.0', 'squeezenet1_0', pretrained=True).eval()

# Example input
x_squeeze = torch.randn(1, 3, 224, 224)

# Forward pass (optional check)
y_squeeze = squeezenet(x_squeeze)
print("SqueezeNet output shape:", y_squeeze.shape)

# Trace and convert to TVM Relay
scripted_squeeze = torch.jit.trace(squeezenet, x_squeeze).eval()
mod_squeeze, params_squeeze = relay.frontend.from_pytorch(scripted_squeeze, [("input0", x_squeeze.shape)])
print("✅ SqueezeNet successfully converted to TVM Relay")
# Choose your target architecture
# Example: x86 CPU
target_str = "llvm -mtriple=aarch64-linux-gnu"
target = tvm.target.Target(target_str)


print(f"Compiling SqueezeNet...")
with tvm.transform.PassContext(opt_level=0):
    lib = relay.build(mod_squeeze, target=target, params=params_squeeze)
# Save compiled library
lib_name = f"SqueezeNet_tvm.so"
lib.export_library(lib_name,
    fcompile=cc.cross_compiler("aarch64-linux-gnu-g++"))

print(f"✅ SqueezeNet compiled and exported as {lib_name}")



print("export ok")

Using cache found in /home/vscode/.cache/torch/hub/pytorch_vision_v0.14.0
/workspaces/TVM-Prep-guide/.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/workspaces/TVM-Prep-guide/.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=SqueezeNet1_0_Weights.IMAGENET1K_V1`. You can also use `weights=SqueezeNet1_0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


SqueezeNet output shape: torch.Size([1, 1000])
✅ SqueezeNet successfully converted to TVM Relay
Compiling SqueezeNet...


One or more operators have not been tuned. Please tune your model for better performance. Use DEBUG logging level to see more details.


✅ SqueezeNet compiled and exported as SqueezeNet_tvm.so
export ok


In [2]:
from PIL import Image
# PyTorch imports
import torch
import torchvision

img_url = "https://github.com/dmlc/mxnet.js/blob/main/data/cat.png?raw=true"
img_path = download_testdata(img_url, "cat.png", module="data")
img = Image.open(img_path).resize((224, 224))

# Preprocess the image and convert to tensor
from torchvision import transforms

my_preprocess = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)
img = my_preprocess(img)
img = np.expand_dims(img, 0)
input_name = "input0"
shape_list = [(input_name, img.shape)]

In [ ]:
from tvm import rpc

#Put here  the PI IP address and the port number of the RPC server
PI_IP = "10.34.2.2" 
PORT = 9090
remote = rpc.connect(PI_IP, PORT)
remote.upload("SqueezeNet_tvm.so")

#Load modlue in PI
rlib = remote.load_module("SqueezeNet_tvm.so")

#create the remote device
dev = remote.cpu(0)

#set up the graph executor
module = graph_executor.GraphModule(rlib["default"](dev))

# Set inputs
module.set_input(input_name, tvm.nd.array(img.astype("float32")))

# Run inference
module.run()

# Get output
output = module.get_output(0)

In [6]:
# Download ImageNet synset labels
synset_url = "".join(
    [
        "https://raw.githubusercontent.com/Cadene/",
        "pretrained-models.pytorch/master/data/",
        "imagenet_synsets.txt",
    ]
)

synset_name = "imagenet_synsets.txt"
synset_path = download_testdata(synset_url, synset_name, module="data")

with open(synset_path) as f:
    synsets = f.readlines()

synsets = [x.strip() for x in synsets]
splits = [line.split(" ") for line in synsets]
key_to_classname = {spl[0]: " ".join(spl[1:]) for spl in splits}

# Download ImageNet class IDs
class_url = "".join(
    [
        "https://raw.githubusercontent.com/Cadene/",
        "pretrained-models.pytorch/master/data/",
        "imagenet_classes.txt",
    ]
)

class_name = "imagenet_classes.txt"
class_path = download_testdata(class_url, class_name, module="data")

with open(class_path) as f:
    class_id_to_key = f.readlines()

class_id_to_key = [x.strip() for x in class_id_to_key]

# Get top-1 result from TVM
top1_tvm = np.argmax(output.numpy()[0])
tvm_class_key = class_id_to_key[top1_tvm]

# Convert input image to PyTorch tensor
torch_img = torch.from_numpy(img)

# Run SqueezeNet locally with PyTorch
with torch.no_grad():
    output = squeezenet(torch_img)

# Get top-1 result from PyTorch
top1_torch = np.argmax(output.numpy())
torch_class_key = class_id_to_key[top1_torch]

# Print results
print(
    "TVM top-1 id: {}, class name: {}".format(
        top1_tvm,
        key_to_classname[tvm_class_key]
    )
)

print(
    "Torch top-1 id: {}, class name: {}".format(
        top1_torch,
        key_to_classname[torch_class_key]
    )
)

TVM top-1 id: 151, class name: Chihuahua
Torch top-1 id: 151, class name: Chihuahua
